In [1]:
import numpy as np   
import pandas as pd # this libary process data
import matplotlib.pyplot as plt    # this 
import tensorflow as tf  #designing neural networks 
from tensorflow.keras import layers, models  # add layers and models
from pathlib import Path  #access directories in system

In [10]:
import pandas as pd

emotions = pd.read_csv('emotions_gray_(32).csv')
emotions.head()

,label,p0,p1,p2,p3,p4,p5,p6,p7,p8,...,p1014,p1015,p1016,p1017,p1018,p1019,p1020,p1021,p1022,p1023
0,0,104,104,105,105,103,99,103,103,110,...,92,149,167,176,160,153,35,43,34,26
1,0,71,100,100,91,116,94,95,98,53,...,77,90,17,19,16,14,21,20,56,17
2,0,126,127,116,113,103,78,46,45,57,...,63,56,72,60,44,38,33,47,75,44
3,0,49,45,44,111,145,78,73,83,93,...,64,55,183,71,137,165,159,149,155,146
4,0,28,29,31,30,69,97,106,123,128,...,137,100,186,29,34,33,35,36,37,37


In [12]:
x = emotions.drop(columns=['label'])
y = emotions['label']

x = x.values.reshape(-1, 32, 32, 1)
x = x.astype('float32') / 255.0
y = tf.keras.utils.to_categorical(y, num_classes=4)


In [13]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

(6520, 32, 32, 1) (1630, 32, 32, 1) (6520, 4) (1630, 4)


In [ ]:
# Add channel dimension -> (N, 28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1) #adding one more dimension at the end 
x_test = np.expand_dims(x_test, axis=-1)

In [26]:
# Hyperparameters
noise_dim = 100    # initial array of 100 size
num_examples_to_generate = 16    
batch_size = 128    # one time 128 size
epochs = 10 

# 2) Build networks
def build_generator():   # a function  with 0 arguments
    # Use 8x8 start so upsampling (×2, ×2) yields 32x32 output to match training images
    model = models.Sequential([
        layers.Dense(8 * 8 * 256, input_shape=(noise_dim,)),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Reshape((8, 8, 256)),
        layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(),
        layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', activation='sigmoid')
    ])
    return model

def build_discriminator():
    model = models.Sequential([
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[32, 32, 1]),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(),
        layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(1)
    ])
    return model

In [27]:
generator = build_generator()
discriminator = build_discriminator()

generator_optimizer = tf.keras.optimizers.Adam(1e-4)
discriminator_optimizer = tf.keras.optimizers.Adam(1e-4)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

c:\Users\Inspector 13\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Inspector 13\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [32]:
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

# 3) Image generation helper
def generate_and_save_images(model, epoch):
    noise = tf.random.normal([num_examples_to_generate, noise_dim])
    generated_images = model(noise, training=False)

    plt.figure(figsize=(4, 4))
    for i in range(generated_images.shape[0]):
        plt.subplot(4, 4, i + 1)
        img = generated_images[i, :, :, 0]
        plt.imshow(img * 255.0, cmap='gray')
        plt.axis('off')
    # Save to a portable path
    save_dir = Path("generated_emotion32_images")
    save_dir.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_dir / f"image_at_epoch_{epoch:04d}.png")
    plt.close()

In [29]:
# 4) Prepare dataset
train_dataset = tf.data.Dataset.from_tensor_slices(x_train)
train_dataset = train_dataset.shuffle(buffer_size=6520).batch(batch_size)

In [33]:
# 5) Training loop
def train(dataset, epochs):
    for epoch in range(epochs):
        for image_batch in dataset:
            curr_batch_size = tf.shape(image_batch)[0]

            # Train discriminator
            noise = tf.random.normal([curr_batch_size, noise_dim])
            with tf.GradientTape() as disc_tape:
                generated_images = generator(noise, training=True)

                real_output = discriminator(image_batch, training=True)
                fake_output = discriminator(generated_images, training=True)

                disc_loss = discriminator_loss(real_output, fake_output)

            grads = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
            discriminator_optimizer.apply_gradients(zip(grads, discriminator.trainable_variables))

            # Train generator
            noise = tf.random.normal([curr_batch_size, noise_dim])
            with tf.GradientTape() as gen_tape:
                generated_images = generator(noise, training=True)
                fake_output = discriminator(generated_images, training=True)
                gen_loss = generator_loss(fake_output)

            grads = gen_tape.gradient(gen_loss, generator.trainable_variables)
            generator_optimizer.apply_gradients(zip(grads, generator.trainable_variables))

        # Logging every epoch
        print(f'Epoch {epoch + 1}, Generator Loss: {gen_loss.numpy():.4f}, Discriminator Loss: {disc_loss.numpy():.4f}')
        generate_and_save_images(generator, epoch + 1)

In [35]:
# Run training
train(train_dataset, 10)

Epoch 1, Generator Loss: 1.1525, Discriminator Loss: 0.8194
Epoch 2, Generator Loss: 0.8597, Discriminator Loss: 1.2796
Epoch 3, Generator Loss: 0.8724, Discriminator Loss: 1.1998
Epoch 4, Generator Loss: 0.6551, Discriminator Loss: 1.7568
Epoch 5, Generator Loss: 0.9987, Discriminator Loss: 1.0056
Epoch 6, Generator Loss: 0.8688, Discriminator Loss: 1.2234
Epoch 7, Generator Loss: 1.0188, Discriminator Loss: 1.0047
Epoch 8, Generator Loss: 1.1576, Discriminator Loss: 1.1591
Epoch 9, Generator Loss: 0.6987, Discriminator Loss: 1.4842
Epoch 10, Generator Loss: 0.7757, Discriminator Loss: 1.2759
